In [2]:
from pymongo import MongoClient

client = MongoClient("127.0.0.1", 27017)
db = client["api_update"]
col = db["java_existent_api_update_instances"]
col.estimated_document_count()

57636

In [3]:
from tqdm import tqdm
import pandas as pd
from packaging.version import Version

tqdm.pandas()
commit_pairs_orig = []

for doc in tqdm(col.find({}), total=col.estimated_document_count()):
    commit = doc["commit"]
    version_before = doc["version_before"]
    version_after = doc["version_after"]
    for pair in doc["api_update_pairs"]:
        old_callee = pair["old_callee"]
        old_api_full_name = old_callee["full_name"]
        if old_callee["parameter_types"] == "":
            old_params = ""
        else:
            old_params = f"({', '.join(old_callee['parameter_types'])})"
        old_api = old_api_full_name + old_params
        old_body = old_callee["body"]
        new_callee = pair["new_callee"]
        new_api_full_name = new_callee["full_name"]
        if new_callee["parameter_types"] == "":
            new_params = ""
        else:
            new_params = f"({', '.join(new_callee['parameter_types'])})"
        new_api = new_api_full_name + new_params
        new_body = new_callee["body"]

        if (len(new_body) == 0) and (len(old_body) > 0):
            continue
        if (len(new_body) > 0) and (len(old_body) == 0):
            continue

        record = [
            doc["package"],
            version_before,
            version_after,
            old_api,
            new_api,
            commit,
        ]
        old_v = Version(version_before)
        new_v = Version(version_after)

        if old_v == new_v:
            continue

        # old_api, -> new_api, up
        # new_api -> old_api, down
        elif old_v < new_v:
            if old_api < new_api:
                rule = [(old_api, new_api), "Up"]
            else:
                rule = [(new_api, old_api), "Down"]

        # old_api -> new_api, down
        # new_api -> old_api, up
        else:
            if old_api < new_api:
                rule = [(old_api, new_api), "Down"]
            else:
                rule = [(new_api, old_api), "Up"]
        commit_pairs_orig.append(record + rule)


commit_pairs_orig = (
    pd.DataFrame(
        commit_pairs_orig,
        columns=[
            "package",
            "version_before",
            "version_after",
            "old_api",
            "new_api",
            "commit",
            "rule",
            "direction",
        ],
    )
    .drop_duplicates()
    .dropna()
)

print(len(commit_pairs_orig), "commit pairs before filtering")
print(
    len(commit_pairs_orig[["package", "rule", "direction"]].drop_duplicates()),
    "rules before filtering",
)
print(
    len(
        commit_pairs_orig[
            ["package", "version_before", "version_after", "old_api", "new_api"]
        ].drop_duplicates()
    ),
    "pairs before filtering",
)

100%|██████████| 57636/57636 [00:13<00:00, 4370.44it/s]


50013 commit pairs before filtering
19961 rules before filtering
35179 pairs before filtering


In [4]:
commit_pairs_orig.head()

,package,version_before,version_after,old_api,new_api,commit,rule,direction
0,junit:junit,4.8.1,4.12,junit.framework.Assert.fail(String),org.junit.Assert.fail(String),00001e41cb1e39747a203bd8a4454f2f1cf9005e,"(junit.framework.Assert.fail(String), org.juni...",Up
1,junit:junit,4.8.1,4.12,junit.framework.Assert.assertNotNull,org.junit.Assert.assertNotNull,00001e41cb1e39747a203bd8a4454f2f1cf9005e,"(junit.framework.Assert.assertNotNull, org.jun...",Up
2,junit:junit,4.8.1,4.12,"junit.framework.Assert.assertEquals(int, int)",org.junit.Assert.assertEquals,00001e41cb1e39747a203bd8a4454f2f1cf9005e,"(junit.framework.Assert.assertEquals(int, int)...",Up
5,junit:junit,4.8.1,4.12,junit.framework.Assert.assertNull,org.junit.Assert.assertNull,00001e41cb1e39747a203bd8a4454f2f1cf9005e,"(junit.framework.Assert.assertNull, org.junit....",Up
9,junit:junit,4.8.1,4.12,"junit.framework.Assert.assertEquals(long, long)","org.junit.Assert.assertEquals(long, long)",00001e41cb1e39747a203bd8a4454f2f1cf9005e,"(junit.framework.Assert.assertEquals(long, lon...",Up


In [5]:
def ratio_fun(row):
    if row["Down"] > row["Up"]:
        row["ratio"] = row["Down"] / row["Up"]
    else:
        row["ratio"] = row["Up"] / row["Down"]

    return row


def find_naive_error_rules():
    rule_df = (
        commit_pairs_orig.groupby(["package", "rule", "direction"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
        .rename_axis(None, axis=1)
    )
    candidate_error_rules = rule_df[(rule_df["Down"] > 0) & (rule_df["Up"] > 0)]
    print(f"{len(candidate_error_rules)} rules have both Up and Down direction")
    candidate_error_rules = candidate_error_rules.apply(ratio_fun, axis=1).sort_values(
        "ratio", ascending=False
    )
    data = []
    for row in candidate_error_rules.itertuples(index=False):
        if row.ratio < 5:
            data.append([row.package, row.rule, "Up"])
            data.append([row.package, row.rule, "Down"])
        else:
            if row.Up > row.Down:
                data.append([row.package, row.rule, "Down"])
            else:
                data.append([row.package, row.rule, "Up"])
    print(len(data), "error rules")
    error_rules = pd.DataFrame(data, columns=["package", "rule", "direction"])
    return error_rules


error_rules = find_naive_error_rules()

436 rules have both Up and Down direction
788 error rules


In [6]:
commit_pairs_filtered = (
    pd.merge(commit_pairs_orig, error_rules, indicator=True, how="left")
    .query('_merge=="left_only"')
    .drop("_merge", axis=1)
)
print(len(commit_pairs_filtered), "commit pairs after filtering")
print(
    len(commit_pairs_filtered[["package", "rule"]].drop_duplicates()),
    "rules after filtering",
)

47714 commit pairs after filtering
19173 rules after filtering


In [7]:
commit_pairs_filtered.head()

,package,version_before,version_after,old_api,new_api,commit,rule,direction
0,junit:junit,4.8.1,4.12,junit.framework.Assert.fail(String),org.junit.Assert.fail(String),00001e41cb1e39747a203bd8a4454f2f1cf9005e,"(junit.framework.Assert.fail(String), org.juni...",Up
1,junit:junit,4.8.1,4.12,junit.framework.Assert.assertNotNull,org.junit.Assert.assertNotNull,00001e41cb1e39747a203bd8a4454f2f1cf9005e,"(junit.framework.Assert.assertNotNull, org.jun...",Up
2,junit:junit,4.8.1,4.12,"junit.framework.Assert.assertEquals(int, int)",org.junit.Assert.assertEquals,00001e41cb1e39747a203bd8a4454f2f1cf9005e,"(junit.framework.Assert.assertEquals(int, int)...",Up
3,junit:junit,4.8.1,4.12,junit.framework.Assert.assertNull,org.junit.Assert.assertNull,00001e41cb1e39747a203bd8a4454f2f1cf9005e,"(junit.framework.Assert.assertNull, org.junit....",Up
4,junit:junit,4.8.1,4.12,"junit.framework.Assert.assertEquals(long, long)","org.junit.Assert.assertEquals(long, long)",00001e41cb1e39747a203bd8a4454f2f1cf9005e,"(junit.framework.Assert.assertEquals(long, lon...",Up


In [8]:
def canonical_pairs(row):
    version_before = row["version_before"]
    version_after = row["version_after"]
    package = row["package"]
    old_api = row["old_api"]
    new_api = row["new_api"]
    commit = row["commit"]
    if Version(version_before) > Version(version_after):
        return pd.Series(
            [package, version_after, version_before, new_api, old_api, commit],
            index=[
                "package",
                "old_version",
                "new_version",
                "old_api",
                "new_api",
                "commit",
            ],
        )
    else:
        return pd.Series(
            [package, version_before, version_after, old_api, new_api, commit],
            index=[
                "package",
                "old_version",
                "new_version",
                "old_api",
                "new_api",
                "commit",
            ],
        )


commit_pairs_full = commit_pairs_filtered.apply(canonical_pairs, axis=1)

In [9]:
commit_pairs_full.head()

,package,old_version,new_version,old_api,new_api,commit
0,junit:junit,4.8.1,4.12,junit.framework.Assert.fail(String),org.junit.Assert.fail(String),00001e41cb1e39747a203bd8a4454f2f1cf9005e
1,junit:junit,4.8.1,4.12,junit.framework.Assert.assertNotNull,org.junit.Assert.assertNotNull,00001e41cb1e39747a203bd8a4454f2f1cf9005e
2,junit:junit,4.8.1,4.12,"junit.framework.Assert.assertEquals(int, int)",org.junit.Assert.assertEquals,00001e41cb1e39747a203bd8a4454f2f1cf9005e
3,junit:junit,4.8.1,4.12,junit.framework.Assert.assertNull,org.junit.Assert.assertNull,00001e41cb1e39747a203bd8a4454f2f1cf9005e
4,junit:junit,4.8.1,4.12,"junit.framework.Assert.assertEquals(long, long)","org.junit.Assert.assertEquals(long, long)",00001e41cb1e39747a203bd8a4454f2f1cf9005e


In [10]:
rule_freq = (
    commit_pairs_full.groupby(["package", "old_api", "new_api"])["commit"]
    .nunique()
    .reset_index()
    .sort_values("commit", ascending=False, ignore_index=True)
)
num_packages = rule_freq["package"].nunique()
num_releases = len(
    pd.concat(
        [
            commit_pairs_full[["package", "old_version"]].rename(
                columns={"old_version": "version"}
            ),
            commit_pairs_full[["package", "new_version"]].rename(
                columns={"new_version": "version"}
            ),
        ]
    ).drop_duplicates()
)
num_rules = len(rule_freq)
num_commits = commit_pairs_full["commit"].nunique()
print(f"# Packages: {num_packages}")
print(f"# Releases: {num_releases}")
print(f"# Rules: {num_rules}")
print(f"# Commits: {num_commits}")

# Packages: 2628
# Releases: 12617
# Rules: 19173
# Commits: 15949


In [11]:
rules_gte10 = rule_freq[rule_freq["commit"] >= 10]
print(
    f"{len(rules_gte10)} rules in {rules_gte10['package'].nunique()} Java packages with freq >= 10"
)
rules_1to10 = rule_freq[(rule_freq["commit"] > 1) & (rule_freq["commit"] < 10)]
print(
    f"{len(rules_1to10)} rules in {rules_1to10['package'].nunique()} Java packages with freq > 1 and < 10"
)
rules_eq1 = rule_freq[rule_freq["commit"] == 1]
print(
    f"{len(rules_eq1)} rules in {rules_eq1['package'].nunique()} Java packages with freq = 1"
)

396 rules in 96 Java packages with freq >= 10
5132 rules in 917 Java packages with freq > 1 and < 10
13645 rules in 2309 Java packages with freq = 1


In [12]:
from utils import cal_sample_size

population_size = len(rules_1to10) + len(rules_eq1)
sample_size = cal_sample_size(population_size)
sample_size_1to10 = round(sample_size * len(rules_1to10) / population_size)
sample_size_eq1 = round(sample_size * len(rules_eq1) / population_size)
print(f"Sample size for all rules with freq < 10: {sample_size}")
print(f"Sample size for rules with freq > 1 and < 10: {sample_size_1to10}")
print(f"Sample size for rules with freq = 1: {sample_size_eq1}")
rules_gte10.to_excel("../benchmark/final/java_api_update_rules_gte10.xlsx")
rules_1to10.sample(sample_size_1to10).to_excel(
    "../benchmark/final/java_api_update_rules_1to10.xlsx", index=False
)
rules_eq1.sample(sample_size_eq1).to_excel(
    "../benchmark/final/java_api_update_rules_eq1.xlsx", index=False
)

Sample size for all rules with freq < 10: 376
Sample size for rules with freq > 1 and < 10: 103
Sample size for rules with freq = 1: 273


In [13]:
rules_gte10_labelled = pd.read_excel(
    "../benchmark/final/java_api_update_rules_gte10-labelled.xlsx",
    keep_default_na=False,
)
correct_rules_gte10 = rules_gte10_labelled[rules_gte10_labelled["correct"] == 1]
print(
    f"{len(rules_gte10_labelled)} rules with freq >= 10, {len(correct_rules_gte10)} are correct"
)
print(f"Accuracy: {len(correct_rules_gte10) / len(rules_gte10_labelled):.3f}")

rules_1to10_labelled = pd.read_excel(
    "../benchmark/final/java_api_update_rules_1to10-labelled.xlsx",
    keep_default_na=False,
)
correct_rules_1to10 = rules_1to10_labelled[rules_1to10_labelled["correct"] == 1]
print(
    f"{len(rules_1to10_labelled)} rules with freq (1, 10), {len(correct_rules_1to10)} are correct"
)
print(f"Accuracy: {len(correct_rules_1to10) / len(rules_1to10_labelled):.3f}")

rules_eq1_labelled = pd.read_excel(
    "../benchmark/final/java_api_update_rules_eq1-labelled.xlsx", keep_default_na=False
)
correct_rules_eq1 = rules_eq1_labelled[rules_eq1_labelled["correct"] == 1]
print(
    f"{len(rules_eq1_labelled)} rules with freq = 1, {len(correct_rules_eq1)} are correct"
)
print(f"Accuracy: {len(correct_rules_eq1) / len(rules_eq1_labelled):.3f}")

rules_exact = pd.concat(
    [correct_rules_gte10, correct_rules_1to10, correct_rules_eq1]
).sort_values("commit", ascending=False, ignore_index=True)
print(
    f"{len(rules_exact)} verified rule in total, {rules_exact['package'].nunique()} packages"
)
total_sampled_rules = pd.concat(
    [rules_gte10_labelled, rules_1to10_labelled, rules_eq1_labelled]
).sort_values("commit", ascending=False, ignore_index=True)
total_sampled_rules.to_csv("../benchmark/final/java_labelled_rules.csv", index=False)

396 rules with freq >= 10, 378 are correct
Accuracy: 0.955
103 rules with freq (1, 10), 90 are correct
Accuracy: 0.874
273 rules with freq = 1, 241 are correct
Accuracy: 0.883
709 verified rule in total, 261 packages


In [14]:
rules_exact

,package,old_api,new_api,commit,correct,evidence
0,junit:junit,"junit.framework.Assert.assertEquals(String, St...",org.junit.Assert.assertEquals,739,1,https://stackoverflow.com/questions/21671527/j...
1,junit:junit,junit.framework.Assert.assertEquals,org.junit.Assert.assertEquals,737,1,https://stackoverflow.com/questions/21671527/j...
2,junit:junit,junit.framework.Assert.assertTrue,org.junit.Assert.assertTrue,735,1,https://stackoverflow.com/questions/21671527/j...
3,junit:junit,"junit.framework.Assert.assertEquals(int, int)",org.junit.Assert.assertEquals,677,1,https://stackoverflow.com/questions/21671527/j...
4,junit:junit,junit.framework.Assert.assertNotNull,org.junit.Assert.assertNotNull,516,1,https://stackoverflow.com/questions/21671527/j...
...,...,...,...,...,...,...
704,com.google.api-ads:adwords-appengine,com.google.api.ads.adwords.jaxws.v201302.cm.Ad...,com.google.api.ads.adwords.jaxws.v201309.cm.Ad...,1,1,https://www.javadoc.io/doc/com.google.api-ads/...
705,com.google.api-ads:adwords-axis,com.google.api.ads.adwords.axis.v201710.cm.Pag...,com.google.api.ads.adwords.axis.v201806.cm.Pag...,1,1,https://www.javadoc.io/doc/com.google.api-ads/...
706,com.google.api-ads:adwords-axis,com.google.api.ads.adwords.axis.v201502.cm.Fee...,com.google.api.ads.adwords.axis.v201506.cm.Fee...,1,1,https://www.javadoc.io/doc/com.google.api-ads/...
707,com.google.api-ads:adwords-axis,com.google.api.ads.adwords.axis.v201502.cm.Con...,com.google.api.ads.adwords.axis.v201506.cm.Con...,1,1,https://www.javadoc.io/doc/com.google.api-ads/...


In [15]:
commit_pairs_exact = (
    rules_exact[["package", "old_api", "new_api"]]
    .merge(commit_pairs_full)
    .drop_duplicates()
)
pairs_exact = commit_pairs_exact[
    ["package", "old_version", "old_api", "new_version", "new_api"]
].drop_duplicates()
print(
    f"{len(commit_pairs_exact)} commit pairs, {len(pairs_exact)} pairs for {len(rules_exact)} correct verified rules"
)
commit_pairs_exact.to_json(
    "../benchmark/final/java_commit_pairs_exact.json", orient="records"
)
pairs_exact.to_json("../benchmark/final/java_api_pairs_exact.json", orient="records")

17051 commit pairs, 6742 pairs for 709 correct verified rules


In [16]:
new_commit_pairs_full = (
    commit_pairs_full.merge(
        total_sampled_rules[total_sampled_rules["correct"] == 0][
            ["package", "old_api", "new_api"]
        ],
        indicator=True,
        how="left",
    )
    .query('_merge=="left_only"')
    .drop("_merge", axis=1)
)
pairs_full = new_commit_pairs_full[
    ["package", "old_version", "old_api", "new_version", "new_api"]
].drop_duplicates()
rules_all = new_commit_pairs_full[["package", "old_api", "new_api"]].drop_duplicates()
print(
    f"{len(new_commit_pairs_full)} commit pairs, {len(pairs_full)} pairs for all {len(rules_all)} rules after removing incorrect verified rules"
)

new_commit_pairs_full.to_json(
    "../benchmark/final/java_commit_pairs_full.json", orient="records"
)
pairs_full.to_json("../benchmark/final/java_api_pairs_full.json", orient="records")

47208 commit pairs, 32518 pairs for all 19110 rules after removing incorrect verified rules


In [17]:
rules_exact.groupby(["package", "old_api"])["new_api"].apply(set).reset_index().to_json(
    "../benchmark/final/java_api_update_rules_exact.json", indent=2, orient="records"
)

In [18]:
pairs_full[["package", "old_api", "new_api"]].drop_duplicates().groupby(
    ["package", "old_api"]
)["new_api"].apply(set).reset_index().to_json(
    "../benchmark/final/java_api_update_rules_full.json", indent=2, orient="records"
)

In [19]:
import pandas as pd

pairs_exact = pd.read_json("../benchmark/final/java_api_pairs_exact.json")
pairs_full = pd.read_json("../benchmark/final/java_api_pairs_full.json")
len(pairs_exact), len(pairs_full)

(6742, 32518)

In [20]:
def get_update_type(row):
    result = ["Major", "Minor", "Patch"]
    old_version = row["old_version"].split(".")
    new_version = row["new_version"].split(".")
    min_len = min(len(old_version), len(new_version))
    row["major"] = (int(old_version[0]), int(new_version[0]))
    for i in range(min_len):
        if old_version[i] != new_version[i]:
            row["update_type"] = result[i]
            return row
    row["update_type"] = result[min_len]
    return row

In [21]:
pairs_exact_sample = (
    pairs_exact.apply(get_update_type, axis=1)
    .groupby(["package", "old_api", "new_api", "major", "update_type"])
    .sample(1)
)[["package", "old_version", "old_api", "new_version", "new_api"]]
print(f"{len(pairs_exact_sample)} sampled pairs in the exact group")
pairs_exact_sample.to_json(
    "../benchmark/final/java_sampled_api_pairs_exact.json", orient="records"
)

pairs_full_sample = (
    pairs_full.apply(get_update_type, axis=1)
    .groupby(["package", "old_api", "new_api", "major", "update_type"])
    .sample(1)
)[["package", "old_version", "old_api", "new_version", "new_api"]]
print(f"{len(pairs_full_sample)} sampled pairs in the full group")
pairs_full_sample.to_json(
    "../benchmark/final/java_sampled_api_pairs_full.json", orient="records"
)

1385 sampled pairs in the exact group
21589 sampled pairs in the full group


In [54]:
import pandas as pd
from tqdm import tqdm

tqdm.pandas()


def sample_instances(group: str):
    instances = pd.read_json(f"../benchmark/final/java_update_instances_{group}.json")
    num_pairs = len(
        instances[
            ["package", "old_api", "new_api", "old_version", "new_version"]
        ].drop_duplicates()
    )
    num_rules = len(instances[["package", "old_api", "new_api"]].drop_duplicates())
    print(f"{group} group:")
    print(f"  # Instances: {len(instances)}")
    print(f"  # Pairs: {num_pairs}")
    print(f"  # Rules: {num_rules}")
    sample = (
        instances.progress_apply(get_update_type, axis=1)
        .groupby(["package", "old_api", "new_api", "major", "update_type"])
        .sample(1, random_state=42)
    )
    print(f"  # Sampled Instances: {len(sample)}")
    sample.to_json(
        f"../benchmark/final/java_sampled_update_instances_{group}.json",
        index=False,
        orient="records",
    )


sample_instances("exact")
sample_instances("full")

exact group:
  # Instances: 82831
  # Pairs: 5369
  # Rules: 573


100%|██████████| 82831/82831 [00:47<00:00, 1756.89it/s]


  # Sampled Instances: 1171
full group:
  # Instances: 128900
  # Pairs: 22022
  # Rules: 12362


100%|██████████| 128900/128900 [01:13<00:00, 1752.97it/s]


  # Sampled Instances: 14193
